<a href="https://colab.research.google.com/github/ilkecerci/cross-domain-turkish-sentiment-analysis/blob/main/notebooks/03_external_validation_dataset_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/Bitirme Projesi/'

train_df = pd.read_csv(base_path + 'train_4domain_natural_v3.csv')
val_df = pd.read_csv(base_path + 'val_4domain_natural_v3.csv')
test_df = pd.read_csv(base_path + 'test_4domain_natural_v3.csv')

existing_texts = set(train_df['text'].dropna().str.lower().str.strip()) | \
                 set(val_df['text'].dropna().str.lower().str.strip()) | \
                 set(test_df['text'].dropna().str.lower().str.strip())

ext_df = pd.read_csv(base_path + 'turkish_movie_sentiment_dataset.csv')
print(f"initial rows in external dataset: {len(ext_df)}")
ext_df['point'] = ext_df['point'].astype(str).str.replace(',', '.').astype(float)
ext_df = ext_df.dropna(subset=['comment', 'point']).copy()
print(f"rows after dropping missing values: {len(ext_df)}")

n_before = len(ext_df)
ext_df = ext_df[(ext_df['point'] <= 2.0) | (ext_df['point'] >= 4.0)].copy()
print(f"rows excluded by rating filter (2 < point < 4): {n_before - len(ext_df)}")
print(f"rows after rating filter: {len(ext_df)}")
ext_df['label'] = ext_df['point'].apply(lambda x: 'Positive' if x >= 4.0 else 'Negative')
ext_df['clean_text'] = ext_df['comment'].astype(str).str.lower().str.strip()

label_counts = ext_df.groupby('clean_text')['label'].nunique()
conflicting_texts = label_counts[label_counts > 1].index
ext_df = ext_df[~ext_df['clean_text'].isin(conflicting_texts)]
print(f"texts with conflicting labels removed: {len(conflicting_texts)}")

ext_df = ext_df.drop_duplicates(subset=['clean_text'], keep='first')
print(f"rows after dropping duplicates and conflicts: {len(ext_df)}")

ext_df_final = ext_df[~ext_df['clean_text'].isin(existing_texts)].copy()
overlap_count = len(ext_df) - len(ext_df_final)

assert ext_df_final['point'].between(2.0, 4.0, inclusive='neither').sum() == 0

print("\n--- external dataset final summary ---")
print(f"overlapping reviews removed: {overlap_count}")
print(f"final total reviews: {len(ext_df_final)}\n")
print("final class distribution:")
print(ext_df_final['label'].value_counts().to_string())

save_path = base_path + 'external_validation_final_v3.csv'
ext_df_final[['comment', 'label']].to_csv(save_path, index=False)
print(f"\nsaved final dataset to {save_path}")

Mounted at /content/drive
initial rows in external dataset: 83227
rows after dropping missing values: 83227
rows excluded by rating filter (2 < point < 4): 25927
rows after rating filter: 57300
texts with conflicting labels removed: 9
rows after dropping duplicates and conflicts: 56722

--- external dataset final summary ---
overlapping reviews removed: 8339
final total reviews: 48383

final class distribution:
label
Positive    40107
Negative     8276

saved final dataset to /content/drive/MyDrive/Bitirme Projesi/external_validation_final_v3.csv
